In [2]:
import json
import os

kaggle_creds = {
    "username":"mohammedelkhayat",
    "key":"a0406c10a888153aa3efc56cf1c24d04"
}

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("✅ Kaggle credentials set!")

✅ Kaggle credentials set!


In [3]:
!pip install -q kaggle

# Download HAM10000 (~3 GB — takes 2-4 minutes)
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data

# Unzip
!unzip -q /content/data/skin-cancer-mnist-ham10000.zip -d /content/data/ham10000
print("✅ Dataset downloaded!")

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:39<00:00, 141MB/s]

✅ Dataset downloaded!


In [4]:
import pandas as pd
import os

base = "/content/data/ham10000"
print("Files:", os.listdir(base))

# Load metadata
meta = pd.read_csv(f"{base}/HAM10000_metadata.csv")
print("\nDataset shape:", meta.shape)
print("\nDisease classes:")
print(meta['dx'].value_counts())
meta.head()

Files: ['hmnist_8_8_L.csv', 'HAM10000_images_part_1', 'ham10000_images_part_1', 'ham10000_images_part_2', 'hmnist_28_28_L.csv', 'HAM10000_metadata.csv', 'hmnist_8_8_RGB.csv', 'hmnist_28_28_RGB.csv', 'HAM10000_images_part_2']

Dataset shape: (10015, 7)

Disease classes:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [5]:
!nvidia-smi

Sun Oct  4 07:08:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import os
import pandas as pd

# --- Config ---
IMG_SIZE = 128
BATCH_SIZE = 32
base = "/content/data/ham10000"

# --- Load metadata ---
meta = pd.read_csv(f"{base}/HAM10000_metadata.csv")

# --- Map image IDs to file paths ---
image_paths = {}
for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    folder_path = f"{base}/{folder}"
    if os.path.exists(folder_path):
        for f in os.listdir(folder_path):
            if f.endswith('.jpg'):
                image_paths[f.split('.')[0]] = os.path.join(folder_path, f)

meta['path'] = meta['image_id'].map(image_paths.get)
meta = meta.dropna(subset=['path'])
print(f"✅ Loaded {len(meta)} images with valid paths")

# --- Encode labels ---
le = LabelEncoder()
meta['label'] = le.fit_transform(meta['dx'])
NUM_CLASSES = len(le.classes_)
print(f"✅ Classes ({NUM_CLASSES}):", list(le.classes_))

# --- Train/validation split ---
train_df, val_df = train_test_split(
    meta, test_size=0.2, stratify=meta['label'], random_state=42
)
print(f"✅ Train: {len(train_df)}  |  Val: {len(val_df)}")

# --- Data generators ---
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True
).flow_from_dataframe(
    train_df, x_col='path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode='raw', batch_size=BATCH_SIZE, shuffle=True
)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
    val_df, x_col='path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode='raw', batch_size=BATCH_SIZE, shuffle=False
)

# --- Build CNN with MobileNetV2 ---
print("\n🧠 Building model...")
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# --- Callbacks ---
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
]

# --- TRAIN ---
print("\n🚀 Starting training...\n")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks
)

print("\n✅ Training complete!")

✅ Loaded 10015 images with valid paths
✅ Classes (7): ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
✅ Train: 8012  |  Val: 2003
Found 8012 validated image filenames.
Found 2003 validated image filenames.

🧠 Building model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


🚀 Starting training...

Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 162s 585ms/step - accuracy: 0.6756 - loss: 0.9691 - val_accuracy: 0.7139 - val_loss: 0.8215 - learning_rate: 0.0010
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 109s 434ms/step - accuracy: 0.7093 - loss: 0.8373 - val_accuracy: 0.7169 - val_loss: 0.7835 - learning_rate: 0.0010
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 110s 438ms/step - accuracy: 0.7162 - loss: 0.8088 - val_accuracy: 0.7359 - val_loss: 0.7445 - learning_rate: 0.0010
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 108s 431ms/step - accuracy: 0.7232 - loss: 0.7858 - val_accuracy: 0.7069 - val_loss: 0.7960 - learning_rate: 0.0010
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 109s 435ms/step - accuracy: 0.7285 - loss: 0.7776 - val_accuracy: 0.7164 - val_loss: 0.7543 - learning_rate: 0.0010
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 108s 431ms/step - accuracy: 0.7371 - loss: 0.7425 - val_accuracy: 0.7349 - val_loss: 0.7429 - learning_rate: 5.0000e-04
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━

In [2]:
import json, os

kaggle_creds = {
    "username": "mohammedelkhayat",
    "key": "PASTE_YOUR_KAGGLE_KEY_HERE"   # ⬅️ your real key
}

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("✅ Kaggle credentials set!")

✅ Kaggle credentials set!


In [3]:
!pip install -q kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data
!unzip -q /content/data/skin-cancer-mnist-ham10000.zip -d /content/data/ham10000
print("✅ Dataset downloaded!")

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [04:20<00:00, 21.4MB/s]

✅ Dataset downloaded!


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np, os, pandas as pd, json

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 5
base = "/content/data/ham10000"

# Load metadata
meta = pd.read_csv(f"{base}/HAM10000_metadata.csv")
image_paths = {}
for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    fp = f"{base}/{folder}"
    if os.path.exists(fp):
        for f in os.listdir(fp):
            if f.endswith('.jpg'):
                image_paths[f.split('.')[0]] = os.path.join(fp, f)

meta['path'] = meta['image_id'].map(image_paths.get)
meta = meta.dropna(subset=['path'])
print(f"✅ Loaded {len(meta)} images")

le = LabelEncoder()
meta['label'] = le.fit_transform(meta['dx'])
NUM_CLASSES = len(le.classes_)
print(f"✅ Classes: {list(le.classes_)}")

train_df, val_df = train_test_split(meta, test_size=0.2, stratify=meta['label'], random_state=42)
print(f"✅ Train: {len(train_df)} | Val: {len(val_df)}")

train_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, zoom_range=0.2,
    width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, vertical_flip=True
).flow_from_dataframe(train_df, x_col='path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), class_mode='raw', batch_size=BATCH_SIZE, shuffle=True)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
    val_df, x_col='path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), class_mode='raw', batch_size=BATCH_SIZE, shuffle=False)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n🚀 Starting training...\n")
history = model.fit(
    train_gen, validation_data=val_gen, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)

print("\n✅ Training complete!")

✅ Loaded 10015 images
✅ Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
✅ Train: 8012 | Val: 2003
Found 8012 validated image filenames.
Found 2003 validated image filenames.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

🚀 Starting training...

Epoch 1/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 134s 487ms/step - accuracy: 0.6717 - loss: 0.9906 - val_accuracy: 0.7074 - val_loss: 0.7958
Epoch 2/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 396ms/step - accuracy: 0.7092 - loss: 0.8365 - val_accuracy: 0.7084 - val_loss: 0.8043
Epoch 3/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 94s 377ms/step - accuracy: 0.7173 - loss: 0.8083 - val_accuracy: 0.7314 - val_loss: 0.7412
Epoch 4/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 91s 364ms/step - accuracy: 0.7212 - loss: 0.7795 - val_accuracy: 0.7354 - val_loss: 0.7513
Epoch 5/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 91s 363ms/step - accuracy: 0.7305 - loss: 0.7645 - val_accuracy: 0.7284 - val_loss: 0.7517

✅ Training complete!


In [5]:
# Save .tflite and label files
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('/content/skin_model.tflite', 'wb') as f:
    f.write(tflite_model)

with open('/content/labels.txt', 'w') as f:
    f.write('\n'.join(le.classes_))

with open('/content/history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f)

print(f"✅ Files saved! Model size: {len(tflite_model)/1024/1024:.2f} MB")

from google.colab import files
files.download('/content/skin_model.tflite')
files.download('/content/labels.txt')
files.download('/content/history.json')
print("📥 Files downloading to your laptop!")

Saved artifact at '/tmp/tmp9u9sitz2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  138076375635472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375636048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375635856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375636432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375635280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375636624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375635664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375637392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375636240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138076375634320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1380763756

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Files downloading to your laptop!
